# Addition transformer in JAX + Flax

Goal: train a small decoder-only transformer on fixed-length character addition examples using only JAX, Flax, and Optax.

Suggested order:
1. Hard-code the tiny vocabulary: digits, space, `+`, and `=`.
2. Generate padded examples like `123+45=168  `.
3. Feed `x = example[:-1]` and train against `y = example[1:]`.
4. Check attention, block, and GPT shapes before training.
5. Add the Optax train state and train loop once the forward pass is stable.

In [1]:
from dataclasses import dataclass
from typing import Optional

import jax
import jax.numpy as jnp
import optax
from flax import linen as nn
from flax.training import train_state

print(jax.default_backend())
print(jax.devices())


/home/ron/stuff/.venv/lib/python3.12/site-packages/jax/_src/cloud_tpu_init.py:88: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


tpu
[TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0)]


In [2]:
# Dataset and token helpers for fixed-width addition examples.

VOCAB = "0123456789 +=P"
PAD = "P"
MAX_NUMBER = 999
FULL_SEQ_LEN = len("999+999=1998 ") + 1
BLOCK_SIZE = FULL_SEQ_LEN - 1

stoi = {ch: i for i, ch in enumerate(VOCAB)}
itos = {i: ch for ch, i in stoi.items()}
pad_token = stoi[PAD]


def encode(text):
  return [stoi[ch] for ch in text]


def decode(tokens):
  return "".join(itos[int(t)] for t in tokens)


def make_addition_example(a, b, *, full_seq_len=FULL_SEQ_LEN):
  text = f"{int(a)}+{int(b)}={int(a) + int(b)} "
  assert len(text) <= full_seq_len
  return encode(text.ljust(full_seq_len, PAD))


def make_addition_dataset(num_samples, rng, *, max_number=MAX_NUMBER, full_seq_len=FULL_SEQ_LEN):
  a_key, b_key = jax.random.split(rng)
  a = jax.device_get(jax.random.randint(a_key, (num_samples,), 0, max_number + 1))
  b = jax.device_get(jax.random.randint(b_key, (num_samples,), 0, max_number + 1))
  examples = jnp.array(
    [make_addition_example(ai, bi, full_seq_len=full_seq_len) for ai, bi in zip(a, b)],
    dtype=jnp.int32,
  )
  x = examples[:, :-1]
  y = examples[:, 1:]

  # Supervise only answer digits and the real trailing space after the answer.
  eq_positions = jnp.argmax(examples == stoi["="], axis=1)
  target_positions = jnp.arange(full_seq_len - 1)[None, :]
  loss_mask = (target_positions >= eq_positions[:, None]) & (y != pad_token)
  return x, y, loss_mask


def get_batch(x_data, y_data, loss_masks, batch_size, rng):
  idx = jax.random.randint(rng, (batch_size,), 0, x_data.shape[0])
  return x_data[idx], y_data[idx], loss_masks[idx]


rng = jax.random.key(0)

batch_size = 2
x_data, y_data, loss_masks = make_addition_dataset(1024, rng)
dummy_tokens, dummy_targets, dummy_loss_mask = get_batch(x_data, y_data, loss_masks, batch_size, rng)
time_steps = dummy_tokens.shape[1]

dummy_tokens.shape, dummy_targets.shape, dummy_loss_mask.shape, decode(dummy_tokens[0]), decode(dummy_targets[0])


((2, 13), (2, 13), (2, 13), '506+678=1184 ', '06+678=1184 P')

In [3]:
import functools
import jax.experimental.pallas as pl
import math

def fused_moe_ffn_kernel(
  x_ref,                 # [MAX_BLOCKS * BLOCK_M, D]
  w_up_ref,              # [E, D, F]
  w_down_ref,            # [E, F, D]
  block_expert_ids_ref,  # [MAX_BLOCKS]
  active_blocks_ref,     # [1]
  y_ref,                 # [MAX_BLOCKS * BLOCK_M, D]
  *,
  MAX_BLOCKS: int,
  D: int,
  F: int,
  BLOCK_M: int,
  BLOCK_F: int,
):
  block_id = pl.program_id(0)

  active_blocks = active_blocks_ref[pl.dslice(0, 1)][0]
  has_work = block_id < active_blocks

  block_expert_ids = block_expert_ids_ref[pl.dslice(0, MAX_BLOCKS)]
  expert_id = jnp.sum(
    jnp.where(
      jnp.arange(MAX_BLOCKS, dtype=block_expert_ids.dtype) == block_id,
      block_expert_ids,
      0,
    )
  )

  def do_work(_):
    token_start = block_id * BLOCK_M

    acc = jnp.zeros((BLOCK_M, D), dtype=jnp.float32)

    x_tile = x_ref[pl.dslice(token_start, BLOCK_M), pl.dslice(0, D)]  # [BLOCK_M, D]

    for f0 in range(0, F, BLOCK_F):
      w_up_tile = w_up_ref[expert_id, pl.dslice(0, D), pl.dslice(f0, BLOCK_F)]  # [D, BLOCK_F]

      h = pl.dot(x_tile, w_up_tile)  # [BLOCK_M, BLOCK_F]

      h = jax.nn.gelu(h)

      w_down_tile = w_down_ref[expert_id, pl.dslice(f0, BLOCK_F), pl.dslice(0, D)]  # [BLOCK_F, D]

      acc += pl.dot(h, w_down_tile)  # [BLOCK_M, D]

    y_ref[pl.dslice(token_start, BLOCK_M), pl.dslice(0, D)] = acc.astype(y_ref.dtype)

  jax.lax.cond(has_work, do_work, lambda _: None, operand=None)


def fused_moe_ffn_pallas(
  x_sorted,
  w_up,
  w_down,
  group_sizes,
  *,
  BLOCK_M=512,
  BLOCK_F=128,
):
  E, D, F = w_up.shape
  M = x_sorted.shape[0]
  if F % BLOCK_F != 0:
    raise ValueError(f"F={F} must be divisible by BLOCK_F={BLOCK_F} for this first fused kernel.")

  max_blocks = math.ceil(M / BLOCK_M) + E
  max_padded_tokens = max_blocks * BLOCK_M

  group_starts = jnp.concatenate([
    jnp.zeros((1,), dtype=group_sizes.dtype),
    jnp.cumsum(group_sizes[:-1]),
  ])
  block_counts = (group_sizes + BLOCK_M - 1) // BLOCK_M
  active_blocks = jnp.sum(block_counts).astype(jnp.int32)
  compact_group_block_starts = jnp.concatenate([
    jnp.zeros((1,), dtype=block_counts.dtype),
    jnp.cumsum(block_counts[:-1]),
  ])
  compact_group_token_starts = compact_group_block_starts * BLOCK_M

  group_starts_sorted = jnp.repeat(
    group_starts,
    group_sizes,
    total_repeat_length=M,
  )
  compact_group_starts_sorted = jnp.repeat(
    compact_group_token_starts,
    group_sizes,
    total_repeat_length=M,
  )
  token_offsets = jnp.arange(M, dtype=jnp.int32) - group_starts_sorted
  compact_offsets = compact_group_starts_sorted + token_offsets
  x_compact = jnp.zeros((max_padded_tokens, D), dtype=x_sorted.dtype).at[compact_offsets].set(x_sorted)

  block_expert_ids = jnp.repeat(
    jnp.arange(E, dtype=jnp.int32),
    block_counts,
    total_repeat_length=max_blocks,
  )

  grid = (max_blocks,)

  kernel = functools.partial(
    fused_moe_ffn_kernel,
    MAX_BLOCKS=max_blocks,
    D=D,
    F=F,
    BLOCK_M=BLOCK_M,
    BLOCK_F=BLOCK_F,
  )

  y_padded = pl.pallas_call(
    kernel,
    out_shape=jax.ShapeDtypeStruct((max_padded_tokens, D), x_sorted.dtype),
    grid=grid,
  )(
    x_compact,
    w_up,
    w_down,
    block_expert_ids,
    jnp.array([active_blocks], dtype=jnp.int32),
  )
  return y_padded[compact_offsets]


def per_expert_moe_ffn_kernel(
  x_ref,             # [EXPERT_STRIDE, D]
  w_up_ref,          # [D, F]
  w_down_ref,        # [F, D]
  group_size_ref,    # [1]
  y_ref,             # [EXPERT_STRIDE, D]
  *,
  D: int,
  F: int,
  BLOCK_M: int,
  BLOCK_F: int,
):
  block_id = pl.program_id(0)
  group_size = group_size_ref[pl.dslice(0, 1)][0]
  has_work = block_id * BLOCK_M < group_size

  def do_work(_):
    token_start = block_id * BLOCK_M
    acc = jnp.zeros((BLOCK_M, D), dtype=jnp.float32)
    x_tile = x_ref[pl.dslice(token_start, BLOCK_M), pl.dslice(0, D)]

    for f0 in range(0, F, BLOCK_F):
      w_up_tile = w_up_ref[pl.dslice(0, D), pl.dslice(f0, BLOCK_F)]
      h = pl.dot(x_tile, w_up_tile)
      h = jax.nn.gelu(h)
      w_down_tile = w_down_ref[pl.dslice(f0, BLOCK_F), pl.dslice(0, D)]
      acc += pl.dot(h, w_down_tile)

    y_ref[pl.dslice(token_start, BLOCK_M), pl.dslice(0, D)] = acc.astype(y_ref.dtype)

  jax.lax.cond(has_work, do_work, lambda _: None, operand=None)


def per_expert_moe_ffn_pallas(
  x_sorted,
  w_up,
  w_down,
  group_sizes,
  *,
  BLOCK_M=256,
  BLOCK_F=128,
):
  E, D, F = w_up.shape
  M = x_sorted.shape[0]
  if F % BLOCK_F != 0:
    raise ValueError(f"F={F} must be divisible by BLOCK_F={BLOCK_F}.")

  blocks_per_expert = math.ceil(M / BLOCK_M)
  expert_stride = blocks_per_expert * BLOCK_M

  group_starts = jnp.concatenate([
    jnp.zeros((1,), dtype=group_sizes.dtype),
    jnp.cumsum(group_sizes[:-1]),
  ])
  expert_ids_sorted = jnp.repeat(
    jnp.arange(E, dtype=jnp.int32),
    group_sizes,
    total_repeat_length=M,
  )
  group_starts_sorted = jnp.repeat(
    group_starts,
    group_sizes,
    total_repeat_length=M,
  )
  local_offsets = jnp.arange(M, dtype=jnp.int32) - group_starts_sorted
  y_sorted = jnp.zeros((M, D), dtype=x_sorted.dtype)

  kernel = functools.partial(
    per_expert_moe_ffn_kernel,
    D=D,
    F=F,
    BLOCK_M=BLOCK_M,
    BLOCK_F=BLOCK_F,
  )

  for expert_id in range(E):
    mask = expert_ids_sorted == expert_id
    x_expert = (
      jnp.zeros((expert_stride, D), dtype=x_sorted.dtype)
      .at[local_offsets]
      .add(jnp.where(mask[:, None], x_sorted, 0))
    )
    y_expert = pl.pallas_call(
      kernel,
      out_shape=jax.ShapeDtypeStruct((expert_stride, D), x_sorted.dtype),
      grid=(blocks_per_expert,),
    )(
      x_expert,
      w_up[expert_id],
      w_down[expert_id],
      jnp.array([group_sizes[expert_id]], dtype=jnp.int32),
    )
    y_sorted = y_sorted + jnp.where(mask[:, None], y_expert[local_offsets], 0)

  return y_sorted


In [4]:
# Flax GPT components for dense and top-1 MoE transformer experiments.

@dataclass(frozen=True)
class GPTConfig:
  vocab_size: int = len(VOCAB)
  block_size: int = BLOCK_SIZE
  n_layer: int = 6
  n_head: int = 6
  n_embd: int = 384
  dropout: float = 0.0
  use_bias: bool = True
  mlp_ratio: int = 4
  dtype: jnp.dtype = jnp.float32
  use_moe: bool = False
  num_experts: int = 4
  top_k: int = 1
  use_fused_moe: bool = False
  fused_moe_backend: str = "compact"

  @property
  def head_dim(self):
    assert self.n_embd % self.n_head == 0
    return self.n_embd // self.n_head


class CausalSelfAttention(nn.Module):
  config: GPTConfig

  @nn.compact
  def __call__(self, x, *, train: bool):
    B, T, C = x.shape
    assert C == self.config.n_embd

    qkv = nn.Dense(3 * C, use_bias=False)(x)

    qkv = qkv.reshape(B, T, 3, self.config.n_head, self.config.head_dim)
    q, k, v = qkv[:, :, 0], qkv[:, :, 1], qkv[:, :, 2]
    q = jnp.transpose(q, (0, 2, 1, 3))
    k = jnp.transpose(k, (0, 2, 1, 3))
    v = jnp.transpose(v, (0, 2, 1, 3))

    mask = jnp.tril(jnp.ones((T, T), dtype=bool))[None, None, :, :]
    scores = q @ jnp.swapaxes(k, -1, -2)
    scores = scores / jnp.sqrt(self.config.head_dim)
    scores = jnp.where(mask, scores, -1e10)

    attn = nn.softmax(scores, axis=-1)
    attn = nn.Dropout(rate=self.config.dropout)(attn, deterministic=not train)

    out = attn @ v
    out = jnp.transpose(out, (0, 2, 1, 3))
    out = out.reshape(B, T, C)
    out = nn.Dense(C, use_bias=False)(out)
    return out


class MLP(nn.Module):
  config: GPTConfig

  @nn.compact
  def __call__(self, x, *, train: bool):
    hidden_dim = self.config.mlp_ratio * self.config.n_embd

    x = nn.Dense(hidden_dim, use_bias=False)(x)
    x = nn.gelu(x)
    x = nn.Dropout(rate=self.config.dropout)(x, deterministic=not train)
    x = nn.Dense(self.config.n_embd, use_bias=False)(x)
    x = nn.Dropout(rate=self.config.dropout)(x, deterministic=not train)
    return x


class MoEMLP(nn.Module):
  config: GPTConfig

  @nn.compact
  def __call__(self, x, *, train: bool):
    if self.config.top_k != 1:
      raise ValueError("MoEMLP currently supports top_k=1 only.")

    B, T, C = x.shape
    E = self.config.num_experts
    H = self.config.mlp_ratio * C

    x_flat = x.reshape(B * T, C) # [M, C]

    router_logits = nn.Dense(E, use_bias=False, name="router")(x_flat) # [M, E]
    router_probs = nn.softmax(router_logits, axis=-1) # [M, E]

    expert_ids = jnp.argmax(router_probs, axis=-1) # [M]
    expert_gates = jnp.max(router_probs, axis=-1) # [M]
    sort_idx = jnp.argsort(expert_ids) # [M]
    unsort_idx = jnp.argsort(sort_idx) # [M]

    x_sorted = x_flat[sort_idx] # [M, C]
    gates_sorted = expert_gates[sort_idx, None] # [M, 1]

    group_sizes = jnp.bincount(expert_ids, length=E).astype(jnp.int32)

    W_up = self.param(
      "expert_w_up",
      nn.initializers.lecun_normal(),
      (E, C, H),
    )
    W_down = self.param(
      "expert_w_down",
      nn.initializers.lecun_normal(),
      (E, H, C),
    )

    if self.config.use_fused_moe:
      if self.config.fused_moe_backend == "compact":
        h = fused_moe_ffn_pallas(x_sorted, W_up, W_down, group_sizes)
      elif self.config.fused_moe_backend == "per_expert":
        h = per_expert_moe_ffn_pallas(x_sorted, W_up, W_down, group_sizes)
      else:
        raise ValueError(f"unknown fused_moe_backend={self.config.fused_moe_backend!r}")
    else:
      h = jax.lax.ragged_dot(x_sorted, W_up, group_sizes)
      h = nn.gelu(h)
      h = jax.lax.ragged_dot(h, W_down, group_sizes)

    h = h * gates_sorted
    h = h[unsort_idx]
    return h.reshape(B, T, C)


class TransformerBlock(nn.Module):
  config: GPTConfig

  @nn.compact
  def __call__(self, x, *, train: bool):
    # GPT-style block: pre-norm attention, residual, pre-norm MLP, residual.
    x = x + CausalSelfAttention(config=self.config)(nn.LayerNorm()(x), train=train)

    if self.config.use_moe:
      ff = MoEMLP(config=self.config)(nn.LayerNorm()(x), train=train)
    else:
      ff = MLP(config=self.config)(nn.LayerNorm()(x), train=train)

    return x + ff


class GPT(nn.Module):
  config: GPTConfig

  @nn.compact
  def __call__(self, idx, *, train: bool):
    _, T = idx.shape
    assert T <= self.config.block_size

    embs = nn.Embed(self.config.vocab_size, self.config.n_embd)(idx)
    pos_emb = nn.Embed(self.config.block_size, self.config.n_embd)(jnp.arange(T))

    h = embs + pos_emb
    h = nn.Dropout(rate=self.config.dropout)(h, deterministic=not train)

    for _ in range(self.config.n_layer):
      h = TransformerBlock(config=self.config)(h, train=train)

    h = nn.LayerNorm()(h)
    return nn.Dense(self.config.vocab_size)(h)


config = GPTConfig(
  # ~10.7M parameters with the current attention/MLP shapes.
  n_layer=6,
  n_head=6,
  n_embd=384,
)
config, config.head_dim, len(VOCAB), BLOCK_SIZE


(GPTConfig(vocab_size=14, block_size=13, n_layer=6, n_head=6, n_embd=384, dropout=0.0, use_bias=True, mlp_ratio=4, dtype=<class 'jax.numpy.float32'>, use_moe=False, num_experts=4, top_k=1, use_fused_moe=False, fused_moe_backend='compact'),
 64,
 14,
 13)

In [5]:
# Evaluation helpers shared by training and scaling-law sweeps.

def count_params(params):
  return sum(x.size for x in jax.tree_util.tree_leaves(params))


def count_active_params(params, config):
  total_params = count_params(params)
  if not getattr(config, "use_moe", False):
    return total_params

  num_experts = getattr(config, "num_experts", 1)
  top_k = getattr(config, "top_k", 1)
  expert_total = 0
  expert_active = 0

  for path, leaf in jax.tree_util.tree_flatten_with_path(params)[0]:
    path_text = "/".join(str(part) for part in path)
    if "expert_w_up" in path_text or "expert_w_down" in path_text:
      expert_total += leaf.size
      expert_active += leaf.size * min(top_k, num_experts) // num_experts

  return total_params - expert_total + expert_active


def cross_entropy_loss(logits, targets, loss_mask=None):
  log_probs = nn.log_softmax(logits, axis=-1)
  target_log_probs = jnp.take_along_axis(
    log_probs,
    indices=targets[..., None],
    axis=-1,
  ).squeeze(-1)
  if loss_mask is None:
    loss_mask = targets != pad_token
  loss_mask = loss_mask.astype(logits.dtype)

  denom = jnp.maximum(loss_mask.sum(), 1)
  return -(target_log_probs * loss_mask).sum() / denom


@jax.jit
def eval_step(state, batch):
  x, y, loss_mask = batch
  logits = state.apply_fn({"params": state.params}, x, train=False)
  return cross_entropy_loss(logits, y, loss_mask)


def evaluate_loss(state, x_eval, y_eval, eval_loss_masks, *, batch_size=256):
  losses = []
  for start in range(0, x_eval.shape[0], batch_size):
    batch = (
      x_eval[start:start + batch_size],
      y_eval[start:start + batch_size],
      eval_loss_masks[start:start + batch_size],
    )
    losses.append(eval_step(state, batch))
  return float(jnp.mean(jnp.array(losses)))


In [ ]:
# Training roadmap for fixed-length addition examples.

@jax.jit
def train_step(state, batch, dropout_rng):
  x, y, loss_mask = batch

  def loss_fn(params):
    logits = state.apply_fn({'params': params}, x, train=True, rngs={'dropout': dropout_rng})
    return cross_entropy_loss(logits, y, loss_mask)

  loss, grads = jax.value_and_grad(loss_fn)(state.params)
  state = state.apply_gradients(grads=grads)
  return state, loss


In [ ]:
# General training roadmap scaffold.

learning_rate = 3e-4
train_batch_size = 256
num_train_samples = 50_000
num_steps = 1_000
log_every = 50

# 1. Build the dataset once. Later, you can split into train/val sets.
rng, data_rng = jax.random.split(rng)
x_train, y_train, train_loss_masks = make_addition_dataset(num_train_samples, data_rng)

# 2. Initialize model parameters from one example batch.
model = GPT(config)
rng, init_rng, dropout_rng = jax.random.split(rng, 3)
init_x, init_y, init_loss_mask = get_batch(
  x_train,
  y_train,
  train_loss_masks,
  train_batch_size,
  init_rng,
)
variables = model.init({'params': init_rng, 'dropout': dropout_rng}, init_x, train=True)

params = variables['params']
total_params = count_params(params)
print(f"Total parameters: {total_params:,}")

# 3. Create optimizer state.
state = train_state.TrainState.create(
  apply_fn=model.apply,
  params=variables['params'],
  tx=optax.adamw(learning_rate),
)

# 4. Training loop shape. Uncomment when you are ready to run.
# losses = []
# for step in range(num_steps):
#   rng, batch_rng, dropout_rng = jax.random.split(rng, 3)
#   batch = get_batch(x_train, y_train, train_loss_masks, train_batch_size, batch_rng)
#   state, loss = train_step(state, batch, dropout_rng)
#   losses.append(loss)

#   if step % log_every == 0:
#     print(f'step {step:04d} loss {float(loss):.4f}')

# state.step, init_x.shape, init_y.shape, init_loss_mask.shape


In [ ]:
def greedy_complete(prompt, *, max_new_tokens=5):
  tokens = encode(prompt)

  for _ in range(max_new_tokens):
    idx = jnp.array(tokens[-config.block_size:], dtype=jnp.int32)[None, :]
    logits = state.apply_fn({'params': state.params}, idx, train=False)
    next_token = int(jnp.argmax(logits[0, -1]))
    tokens.append(next_token)

    if itos[next_token] in {" ", PAD}:
      break

  return decode(tokens)


prompt = "506+678="
expected = "506+678=1184 "
completion = greedy_complete(prompt)

print("prompt:    ", repr(prompt))
print("expected:  ", repr(expected))
print("completed: ", repr(completion))

In [ ]:
# Chinchilla-style trial wrapper: vary config and data size, record loss metrics.

def run_chinchilla_trial(
  run_config,
  num_train_samples,
  *,
  num_steps=None,
  batch_size=256,
  learning_rate=3e-4,
  num_val_samples=4096,
  seed=0,
  val_seed=None,
  log_every=None,
):
  rng = jax.random.key(seed)
  if val_seed is None:
    rng, train_data_rng, val_data_rng = jax.random.split(rng, 3)
  else:
    rng, train_data_rng = jax.random.split(rng)
    val_data_rng = jax.random.key(val_seed)
  x_train, y_train, train_masks = make_addition_dataset(num_train_samples, train_data_rng)
  x_val, y_val, val_masks = make_addition_dataset(num_val_samples, val_data_rng)

  if num_steps is None:
    num_steps = max(1, num_train_samples // batch_size)

  model = GPT(run_config)
  rng, init_rng, dropout_rng = jax.random.split(rng, 3)
  init_batch = get_batch(x_train, y_train, train_masks, batch_size, init_rng)
  variables = model.init({'params': init_rng, 'dropout': dropout_rng}, init_batch[0], train=True)

  state = train_state.TrainState.create(
    apply_fn=model.apply,
    params=variables['params'],
    tx=optax.adamw(learning_rate),
  )

  params_total = count_params(state.params)
  params_active = count_active_params(state.params, run_config)
  last_loss = None
  train_tokens_seen = 0

  for step in range(num_steps):
    rng, batch_rng, dropout_rng = jax.random.split(rng, 3)
    batch = get_batch(x_train, y_train, train_masks, batch_size, batch_rng)
    state, loss = train_step(state, batch, dropout_rng)
    train_tokens_seen += int(batch[2].sum())
    last_loss = loss

    if log_every is not None and step % log_every == 0:
      loss_value = float(loss)
      print(f'step {step:04d} loss {loss_value:.4f}')

  train_eval_loss = evaluate_loss(state, x_train[:num_val_samples], y_train[:num_val_samples], train_masks[:num_val_samples], batch_size=batch_size)
  val_loss = evaluate_loss(state, x_val, y_val, val_masks, batch_size=batch_size)

  metrics = {
    'N': params_active,
    'N_total': params_total,
    'N_active': params_active,
    'D': train_tokens_seen,
    'C': 6 * params_active * train_tokens_seen,
    'train_loss': train_eval_loss,
    'val_loss': val_loss,
  }
  return state, metrics


In [ ]:
# Constant-LR sweep for fitting Chinchilla-style laws.
# Writes to a new file so the first harness run stays untouched.
import json
import math
from pathlib import Path

metrics_path = Path("outputs/chinchilla_constant_lr_metrics.json")


def load_sweep_metrics(path=metrics_path):
  if path.exists():
    return json.loads(path.read_text())
  return {}


def save_sweep_metrics(metrics, path=metrics_path):
  path.write_text(json.dumps(metrics, indent=2, sort_keys=True))


def lr_tag(learning_rate):
  return f"{learning_rate:.0e}".replace("-", "m")


def sweep_key(config_idx, num_samples, epochs, learning_rate):
  return f"config_{config_idx}_samples_{num_samples}_epochs_{epochs}_lr_{lr_tag(learning_rate)}"

sweep_configs = [
  {"config": GPTConfig(n_layer=1, n_head=2, n_embd=64), "learning_rate": 3e-4},
  {"config": GPTConfig(n_layer=2, n_head=4, n_embd=128), "learning_rate": 3e-4},
  {"config": GPTConfig(n_layer=3, n_head=4, n_embd=192), "learning_rate": 2e-4},
  {"config": GPTConfig(n_layer=4, n_head=4, n_embd=256), "learning_rate": 2e-4},
  {"config": GPTConfig(n_layer=6, n_head=6, n_embd=384), "learning_rate": 1e-4},
]

sweep_sample_sizes = [
  2_000,
  10_000,
  50_000,
  200_000,
]

sweep_metrics = load_sweep_metrics()
sweep_epochs_grid = [3, 10]
sweep_batch_size = 256
sweep_num_val_samples = 8192
sweep_val_seed = 12345
sweep_log_every = 100

# Constant LR only. The largest model gets a lower LR because the diagnostic
# sweep showed 3e-4 constant under-optimizes it at short horizons.
for config_idx, config_spec in enumerate(sweep_configs):
  trial_config = config_spec["config"]
  learning_rate = config_spec["learning_rate"]
  for sample_idx, num_samples in enumerate(sweep_sample_sizes):
    for epochs in sweep_epochs_grid:
      key = sweep_key(config_idx, num_samples, epochs, learning_rate)
      if key in sweep_metrics:
        metrics = sweep_metrics[key]
        metrics.setdefault("N_total", metrics["N"])
        metrics.setdefault("N_active", metrics["N"])
        metrics["N"] = metrics["N_active"]
        metrics["C"] = 6 * metrics["N_active"] * metrics["D"]
        save_sweep_metrics(sweep_metrics)
        print(f"skipping completed {key}")
        continue

      num_steps = epochs * math.ceil(num_samples / sweep_batch_size)
      seed = 10_000 * config_idx + 100 * sample_idx + epochs
      print(f"running {key}: steps={num_steps}")
      trial_state, metrics = run_chinchilla_trial(
        trial_config,
        num_train_samples=num_samples,
        num_steps=num_steps,
        batch_size=sweep_batch_size,
        learning_rate=learning_rate,
        num_val_samples=sweep_num_val_samples,
        seed=seed,
        val_seed=sweep_val_seed,
        log_every=sweep_log_every,
      )
      metrics.update({
        "config_idx": config_idx,
        "epochs": epochs,
        "learning_rate": learning_rate,
        "num_train_samples": num_samples,
        "num_steps": num_steps,
        "val_seed": sweep_val_seed,
      })
      sweep_metrics[key] = metrics
      save_sweep_metrics(sweep_metrics)
      print(key, metrics)
      del trial_state

best_by_compute = sorted(sweep_metrics.items(), key=lambda item: item[1]["C"])
best_so_far = None
for key, metrics in best_by_compute:
  if best_so_far is None or metrics["val_loss"] < best_so_far[1]["val_loss"]:
    best_so_far = (key, metrics)
    print("frontier", key, "C", metrics["C"], "N", metrics["N"], "D", metrics["D"], "val", metrics["val_loss"])

# sweep_metrics


In [ ]:
# Constant-LR MoE sweep for comparing dense vs sparse scaling.
# Uses the same run_chinchilla_trial helper, which records both N_total and N_active.

moe_metrics_path = Path("outputs/chinchilla_moe_metrics.json")
moe_metrics_path.parent.mkdir(parents=True, exist_ok=True)


def load_moe_sweep_metrics(path=moe_metrics_path):
  if path.exists():
    return json.loads(path.read_text())
  return {}


def save_moe_sweep_metrics(metrics, path=moe_metrics_path):
  path.write_text(json.dumps(metrics, indent=2, sort_keys=True))


def moe_sweep_key(config_idx, num_samples, epochs, learning_rate, num_experts, top_k):
  return (
    f"moe_config_{config_idx}_experts_{num_experts}_topk_{top_k}_"
    f"samples_{num_samples}_epochs_{epochs}_lr_{lr_tag(learning_rate)}"
  )


moe_sweep_configs = [
  {"config": GPTConfig(n_layer=1, n_head=2, n_embd=64, use_moe=True, num_experts=4, top_k=1), "learning_rate": 3e-4},
  {"config": GPTConfig(n_layer=2, n_head=4, n_embd=128, use_moe=True, num_experts=4, top_k=1), "learning_rate": 3e-4},
  {"config": GPTConfig(n_layer=3, n_head=4, n_embd=192, use_moe=True, num_experts=4, top_k=1), "learning_rate": 2e-4},
  {"config": GPTConfig(n_layer=4, n_head=4, n_embd=256, use_moe=True, num_experts=4, top_k=1), "learning_rate": 2e-4},
  {"config": GPTConfig(n_layer=6, n_head=6, n_embd=384, use_moe=True, num_experts=4, top_k=1), "learning_rate": 1e-4},
]

moe_sweep_sample_sizes = sweep_sample_sizes
moe_sweep_epochs_grid = sweep_epochs_grid
moe_sweep_batch_size = sweep_batch_size
moe_sweep_num_val_samples = sweep_num_val_samples
moe_sweep_val_seed = sweep_val_seed
moe_sweep_log_every = sweep_log_every
moe_sweep_metrics = load_moe_sweep_metrics()

for config_idx, config_spec in enumerate(moe_sweep_configs):
  trial_config = config_spec["config"]
  learning_rate = config_spec["learning_rate"]
  for sample_idx, num_samples in enumerate(moe_sweep_sample_sizes):
    for epochs in moe_sweep_epochs_grid:
      key = moe_sweep_key(
        config_idx,
        num_samples,
        epochs,
        learning_rate,
        trial_config.num_experts,
        trial_config.top_k,
      )
      if key in moe_sweep_metrics:
        print(f"skipping completed {key}")
        continue

      num_steps = epochs * math.ceil(num_samples / moe_sweep_batch_size)
      seed = 50_000 + 10_000 * config_idx + 100 * sample_idx + epochs
      print(f"running {key}: steps={num_steps}")
      trial_state, metrics = run_chinchilla_trial(
        trial_config,
        num_train_samples=num_samples,
        num_steps=num_steps,
        batch_size=moe_sweep_batch_size,
        learning_rate=learning_rate,
        num_val_samples=moe_sweep_num_val_samples,
        seed=seed,
        val_seed=moe_sweep_val_seed,
        log_every=moe_sweep_log_every,
      )
      metrics.update({
        "architecture": "moe",
        "config_idx": config_idx,
        "epochs": epochs,
        "learning_rate": learning_rate,
        "num_train_samples": num_samples,
        "num_steps": num_steps,
        "num_experts": trial_config.num_experts,
        "top_k": trial_config.top_k,
        "n_layer": trial_config.n_layer,
        "n_head": trial_config.n_head,
        "n_embd": trial_config.n_embd,
        "val_seed": moe_sweep_val_seed,
      })
      moe_sweep_metrics[key] = metrics
      save_moe_sweep_metrics(moe_sweep_metrics)
      print(key, metrics)
      del trial_state

best_by_active_compute = sorted(moe_sweep_metrics.items(), key=lambda item: item[1]["C"])
best_so_far = None
for key, metrics in best_by_active_compute:
  if best_so_far is None or metrics["val_loss"] < best_so_far[1]["val_loss"]:
    best_so_far = (key, metrics)
    print(
      "moe frontier",
      key,
      "C_active", metrics["C"],
      "N_active", metrics["N_active"],
      "N_total", metrics["N_total"],
      "D", metrics["D"],
      "val", metrics["val_loss"],
    )

# moe_sweep_metrics


In [ ]:
# Fit a Chinchilla-style loss law from the constant-LR sweep.
# We fit L(N, D) = E + A / N^alpha + B / D^beta, then solve for
# compute-optimal N and D under C = 6ND.

import json
import math
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import differential_evolution, least_squares

fit_metrics_path = Path("outputs/chinchilla_constant_lr_metrics.json")
fit_target = "val_loss"
fit_epochs = 10  # Use the more optimized comparable slice first.
fit_log_residuals = True


def load_chinchilla_rows(path=fit_metrics_path):
  raw = json.loads(path.read_text())
  pattern = re.compile(r"config_(\d+)_samples_(\d+)_epochs_(\d+)_lr_(.+)")
  rows = []

  for key, metrics in raw.items():
    match = pattern.match(key)
    row = dict(metrics)
    row["key"] = key
    if match is not None:
      row.setdefault("config_idx", int(match.group(1)))
      row.setdefault("num_train_samples", int(match.group(2)))
      row.setdefault("epochs", int(match.group(3)))
    rows.append(row)

  return rows


def chinchilla_loss(theta, N, D):
  log_A, alpha, log_B, beta, log_E = theta
  A = np.exp(log_A)
  B = np.exp(log_B)
  E = np.exp(log_E)
  return E + A * np.power(N, -alpha) + B * np.power(D, -beta)


def unpack_chinchilla_theta(theta):
  log_A, alpha, log_B, beta, log_E = theta
  return {
    "A": float(np.exp(log_A)),
    "alpha": float(alpha),
    "B": float(np.exp(log_B)),
    "beta": float(beta),
    "E": float(np.exp(log_E)),
  }


def fit_chinchilla_law(rows, *, target=fit_target, log_residuals=fit_log_residuals):
  N = np.array([row["N"] for row in rows], dtype=float)
  D = np.array([row["D"] for row in rows], dtype=float)
  y = np.array([row[target] for row in rows], dtype=float)

  if np.any(y <= 0):
    raise ValueError("loss values must be positive for this fit")

  bounds = [
    (-30.0, 90.0),              # log_A
    (0.01, 4.0),                # alpha
    (-30.0, 90.0),              # log_B
    (0.01, 4.0),                # beta
    (-40.0, math.log(y.min() * 0.95)),  # log_E, below the best observed loss
  ]

  def residuals(theta):
    pred = chinchilla_loss(theta, N, D)
    if log_residuals:
      return np.log(pred) - np.log(y)
    return pred - y

  def objective(theta):
    r = residuals(theta)
    return float(np.mean(r * r))

  # Differential evolution gives the nonlinear fit a stable starting point;
  # least_squares then polishes the local optimum.
  global_fit = differential_evolution(
    objective,
    bounds=bounds,
    seed=0,
    maxiter=1000,
    tol=1e-8,
    polish=False,
  )
  local_fit = least_squares(
    residuals,
    global_fit.x,
    bounds=([lo for lo, _ in bounds], [hi for _, hi in bounds]),
    max_nfev=50_000,
  )

  theta = local_fit.x
  pred = chinchilla_loss(theta, N, D)
  params = unpack_chinchilla_theta(theta)

  alpha = params["alpha"]
  beta = params["beta"]
  A = params["A"]
  B = params["B"]
  n_exponent = beta / (alpha + beta)
  d_exponent = alpha / (alpha + beta)
  G = (alpha * A / (beta * B)) ** (1.0 / (alpha + beta))

  diagnostics = {
    "rmse": float(np.sqrt(np.mean((pred - y) ** 2))),
    "log_rmse": float(np.sqrt(np.mean((np.log(pred) - np.log(y)) ** 2))),
    "mape_pct": float(np.mean(np.abs((pred - y) / y)) * 100.0),
    "N_exponent": float(n_exponent),
    "D_exponent": float(d_exponent),
    "G": float(G),
    "N_prefactor": float(G),
    "D_prefactor": float(1.0 / G),
  }

  fitted_rows = []
  for row, actual, predicted in zip(rows, y, pred):
    fitted = dict(row)
    fitted["pred_loss"] = float(predicted)
    fitted["loss_ratio_pred_actual"] = float(predicted / actual)
    fitted_rows.append(fitted)

  return params, diagnostics, fitted_rows


def compute_frontier(rows, *, target=fit_target):
  frontier = []
  best = None
  for row in sorted(rows, key=lambda r: r["C"]):
    if best is None or row[target] < best[target]:
      best = row
      frontier.append(row)
  return frontier


all_rows = load_chinchilla_rows()
fit_rows = [row for row in all_rows if row.get("epochs") == fit_epochs]

params, diagnostics, fitted_rows = fit_chinchilla_law(fit_rows)
frontier_rows = compute_frontier(fit_rows)

print(f"fit rows: {len(fit_rows)} from epochs={fit_epochs}")
print("\nLoss law:")
print(
  "L(N, D) = "
  f"{params['E']:.6g} + {params['A']:.6g} / N^{params['alpha']:.4f} "
  f"+ {params['B']:.6g} / D^{params['beta']:.4f}"
)

print("\nCompute-optimal laws, using C = 6ND:")
print(
  f"N_opt(C) = {diagnostics['N_prefactor']:.6g} * (C / 6)^{diagnostics['N_exponent']:.4f}"
)
print(
  f"D_opt(C) = {diagnostics['D_prefactor']:.6g} * (C / 6)^{diagnostics['D_exponent']:.4f}"
)
print(f"D/N exponent gap = {diagnostics['D_exponent'] - diagnostics['N_exponent']:.4f}")

print("\nFit diagnostics:")
print(
  f"RMSE={diagnostics['rmse']:.6g}, "
  f"log_RMSE={diagnostics['log_rmse']:.4f}, "
  f"MAPE={diagnostics['mape_pct']:.2f}%"
)

print("\nObserved validation-loss frontier within the fitted slice:")
for row in frontier_rows:
  print(
    f"C={row['C']:.3e} N={row['N']:.3g} D={row['D']:.3g} "
    f"val={row[fit_target]:.6g} key={row['key']}"
  )

print("\nWorst residuals:")
for row in sorted(fitted_rows, key=lambda r: abs(math.log(r["loss_ratio_pred_actual"])), reverse=True)[:8]:
  print(
    f"{row['key']}: actual={row[fit_target]:.6g}, "
    f"pred={row['pred_loss']:.6g}, ratio={row['loss_ratio_pred_actual']:.2f}"
  )

print("\nRecommended N,D at observed frontier compute budgets:")
for row in frontier_rows:
  C_over_6 = row["C"] / 6.0
  N_opt = diagnostics["N_prefactor"] * (C_over_6 ** diagnostics["N_exponent"])
  D_opt = diagnostics["D_prefactor"] * (C_over_6 ** diagnostics["D_exponent"])
  print(f"C={row['C']:.3e}: N_opt={N_opt:.3g}, D_opt={D_opt:.3g}, D/N={D_opt / N_opt:.3g}")

actual = np.array([row[fit_target] for row in fitted_rows], dtype=float)
predicted = np.array([row["pred_loss"] for row in fitted_rows], dtype=float)
compute = np.array([row["C"] for row in fitted_rows], dtype=float)
ratios = predicted / actual

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].scatter(actual, predicted)
lo = min(actual.min(), predicted.min())
hi = max(actual.max(), predicted.max())
axes[0].plot([lo, hi], [lo, hi], color="black", linewidth=1)
axes[0].set_xscale("log")
axes[0].set_yscale("log")
axes[0].set_xlabel("actual validation loss")
axes[0].set_ylabel("predicted validation loss")
axes[0].set_title("Observed vs predicted")

axes[1].scatter(compute, ratios)
axes[1].axhline(1.0, color="black", linewidth=1)
axes[1].set_xscale("log")
axes[1].set_xlabel("compute C")
axes[1].set_ylabel("predicted / actual")
axes[1].set_title("Residual ratio by compute")

plt.tight_layout()
plt.show()


In [ ]:
# Fit Chinchilla-style laws for the MoE sweep.
# We fit two views:
#   1. active-compute law: L(N_active, D), with C_active = 6 * N_active * D
#   2. capacity law:       L(N_total, D), to see whether stored expert capacity explains loss

import json
import math
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import differential_evolution, least_squares

moe_fit_metrics_path = Path("outputs/chinchilla_moe_metrics.json")
dense_fit_metrics_path = Path("outputs/chinchilla_constant_lr_metrics.json")
moe_fit_target = "val_loss"
moe_fit_epochs = 10
moe_fit_log_residuals = True


def load_dense_or_moe_rows(path, *, architecture):
  raw = json.loads(path.read_text())
  dense_pattern = re.compile(r"config_(\d+)_samples_(\d+)_epochs_(\d+)_lr_(.+)")
  moe_pattern = re.compile(r"moe_config_(\d+)_experts_(\d+)_topk_(\d+)_samples_(\d+)_epochs_(\d+)_lr_(.+)")
  rows = []

  for key, metrics in raw.items():
    row = dict(metrics)
    row["key"] = key
    row["architecture"] = architecture

    moe_match = moe_pattern.match(key)
    dense_match = dense_pattern.match(key)
    if moe_match is not None:
      row.setdefault("config_idx", int(moe_match.group(1)))
      row.setdefault("num_experts", int(moe_match.group(2)))
      row.setdefault("top_k", int(moe_match.group(3)))
      row.setdefault("num_train_samples", int(moe_match.group(4)))
      row.setdefault("epochs", int(moe_match.group(5)))
    elif dense_match is not None:
      row.setdefault("config_idx", int(dense_match.group(1)))
      row.setdefault("num_train_samples", int(dense_match.group(2)))
      row.setdefault("epochs", int(dense_match.group(3)))

    row.setdefault("N_active", row.get("N"))
    row.setdefault("N_total", row.get("N"))
    row.setdefault("C_active", 6 * row["N_active"] * row["D"])
    rows.append(row)

  return rows


def chinchilla_loss(theta, N, D):
  log_A, alpha, log_B, beta, log_E = theta
  A = np.exp(log_A)
  B = np.exp(log_B)
  E = np.exp(log_E)
  return E + A * np.power(N, -alpha) + B * np.power(D, -beta)


def unpack_theta(theta):
  log_A, alpha, log_B, beta, log_E = theta
  return {
    "A": float(np.exp(log_A)),
    "alpha": float(alpha),
    "B": float(np.exp(log_B)),
    "beta": float(beta),
    "E": float(np.exp(log_E)),
  }


def fit_law(rows, *, n_key, target=moe_fit_target, log_residuals=moe_fit_log_residuals, seed=0):
  N = np.array([row[n_key] for row in rows], dtype=float)
  D = np.array([row["D"] for row in rows], dtype=float)
  y = np.array([row[target] for row in rows], dtype=float)

  if np.any(y <= 0):
    raise ValueError("loss values must be positive for this fit")

  bounds = [
    (-30.0, 90.0),
    (0.01, 4.0),
    (-30.0, 90.0),
    (0.01, 4.0),
    (-40.0, math.log(y.min() * 0.95)),
  ]

  def residuals(theta):
    pred = chinchilla_loss(theta, N, D)
    if log_residuals:
      return np.log(pred) - np.log(y)
    return pred - y

  def objective(theta):
    r = residuals(theta)
    return float(np.mean(r * r))

  global_fit = differential_evolution(
    objective,
    bounds=bounds,
    seed=seed,
    maxiter=1000,
    tol=1e-8,
    polish=False,
  )
  local_fit = least_squares(
    residuals,
    global_fit.x,
    bounds=([lo for lo, _ in bounds], [hi for _, hi in bounds]),
    max_nfev=50_000,
  )

  theta = local_fit.x
  pred = chinchilla_loss(theta, N, D)
  params = unpack_theta(theta)
  alpha = params["alpha"]
  beta = params["beta"]
  A = params["A"]
  B = params["B"]

  n_exponent = beta / (alpha + beta)
  d_exponent = alpha / (alpha + beta)
  G = (alpha * A / (beta * B)) ** (1.0 / (alpha + beta))

  diagnostics = {
    "rmse": float(np.sqrt(np.mean((pred - y) ** 2))),
    "log_rmse": float(np.sqrt(np.mean((np.log(pred) - np.log(y)) ** 2))),
    "mape_pct": float(np.mean(np.abs((pred - y) / y)) * 100.0),
    "N_exponent": float(n_exponent),
    "D_exponent": float(d_exponent),
    "N_prefactor": float(G),
    "D_prefactor": float(1.0 / G),
  }

  fitted_rows = []
  for row, actual, predicted in zip(rows, y, pred):
    fitted = dict(row)
    fitted["fit_n_key"] = n_key
    fitted["pred_loss"] = float(predicted)
    fitted["loss_ratio_pred_actual"] = float(predicted / actual)
    fitted_rows.append(fitted)

  return params, diagnostics, fitted_rows


def compute_frontier(rows, *, compute_key="C", target=moe_fit_target):
  frontier = []
  best = None
  for row in sorted(rows, key=lambda r: r[compute_key]):
    if best is None or row[target] < best[target]:
      best = row
      frontier.append(row)
  return frontier


def print_law(label, params, diagnostics, *, compute_label="C_active"):
  print(f"\n{label}")
  print(
    "L(N, D) = "
    f"{params['E']:.6g} + {params['A']:.6g} / N^{params['alpha']:.4f} "
    f"+ {params['B']:.6g} / D^{params['beta']:.4f}"
  )
  print(
    f"N_opt({compute_label}) = {diagnostics['N_prefactor']:.6g} "
    f"* ({compute_label} / 6)^{diagnostics['N_exponent']:.4f}"
  )
  print(
    f"D_opt({compute_label}) = {diagnostics['D_prefactor']:.6g} "
    f"* ({compute_label} / 6)^{diagnostics['D_exponent']:.4f}"
  )
  print(
    f"RMSE={diagnostics['rmse']:.6g}, "
    f"log_RMSE={diagnostics['log_rmse']:.4f}, "
    f"MAPE={diagnostics['mape_pct']:.2f}%"
  )


dense_rows_all = load_dense_or_moe_rows(dense_fit_metrics_path, architecture="dense")
moe_rows_all = load_dense_or_moe_rows(moe_fit_metrics_path, architecture="moe")

dense_rows = [row for row in dense_rows_all if row.get("epochs") == moe_fit_epochs]
moe_rows = [row for row in moe_rows_all if row.get("epochs") == moe_fit_epochs]

active_params, active_diag, active_fit_rows = fit_law(moe_rows, n_key="N_active", seed=1)
total_params, total_diag, total_fit_rows = fit_law(moe_rows, n_key="N_total", seed=2)

dense_frontier = compute_frontier(dense_rows, compute_key="C")
moe_frontier = compute_frontier(moe_rows, compute_key="C")
combined_frontier = compute_frontier(dense_rows + moe_rows, compute_key="C")

print(f"MoE fit rows: {len(moe_rows)} from epochs={moe_fit_epochs}")
print_law("MoE active-parameter law: L(N_active, D)", active_params, active_diag, compute_label="C_active")
print_law("MoE total-parameter capacity law: L(N_total, D)", total_params, total_diag, compute_label="C_total_proxy")

print("\nMoE observed active-compute frontier:")
for row in moe_frontier:
  print(
    f"C={row['C']:.3e} N_active={row['N_active']:.3g} N_total={row['N_total']:.3g} "
    f"D={row['D']:.3g} val={row[moe_fit_target]:.6g} key={row['key']}"
  )

print("\nCombined dense/MoE active-compute frontier:")
for row in combined_frontier:
  print(
    f"C={row['C']:.3e} arch={row['architecture']} cfg={row.get('config_idx')} "
    f"N_active={row['N_active']:.3g} N_total={row['N_total']:.3g} "
    f"D={row['D']:.3g} val={row[moe_fit_target]:.6g}"
  )

print("\nWorst MoE active-law residuals:")
for row in sorted(active_fit_rows, key=lambda r: abs(math.log(r["loss_ratio_pred_actual"])), reverse=True)[:8]:
  print(
    f"{row['key']}: actual={row[moe_fit_target]:.6g}, "
    f"pred={row['pred_loss']:.6g}, ratio={row['loss_ratio_pred_actual']:.2f}"
  )

print("\nActive-law recommended N_active,D at MoE frontier budgets:")
for row in moe_frontier:
  C_over_6 = row["C"] / 6.0
  N_opt = active_diag["N_prefactor"] * (C_over_6 ** active_diag["N_exponent"])
  D_opt = active_diag["D_prefactor"] * (C_over_6 ** active_diag["D_exponent"])
  print(f"C={row['C']:.3e}: N_active_opt={N_opt:.3g}, D_opt={D_opt:.3g}, D/N_active={D_opt / N_opt:.3g}")

# Plot 1: MoE observed vs predicted for active and total parameter views.
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, fitted_rows, title in [
  (axes[0], active_fit_rows, "MoE active-param fit"),
  (axes[1], total_fit_rows, "MoE total-param fit"),
]:
  actual = np.array([row[moe_fit_target] for row in fitted_rows], dtype=float)
  predicted = np.array([row["pred_loss"] for row in fitted_rows], dtype=float)
  ax.scatter(actual, predicted)
  lo = min(actual.min(), predicted.min())
  hi = max(actual.max(), predicted.max())
  ax.plot([lo, hi], [lo, hi], color="black", linewidth=1)
  ax.set_xscale("log")
  ax.set_yscale("log")
  ax.set_xlabel("actual validation loss")
  ax.set_ylabel("predicted validation loss")
  ax.set_title(title)

for label, rows in [("dense", dense_frontier), ("moe", moe_frontier), ("combined", combined_frontier)]:
  axes[2].plot(
    [row["C"] for row in rows],
    [row[moe_fit_target] for row in rows],
    marker="o",
    label=label,
  )
axes[2].set_xscale("log")
axes[2].set_yscale("log")
axes[2].set_xlabel("active compute C")
axes[2].set_ylabel("best validation loss so far")
axes[2].set_title("Active-compute frontiers")
axes[2].legend()

plt.tight_layout()
plt.show()

# Plot 2: matched dense/MoE validation losses by config at the fitted epoch slice.
matched_pairs = []
for dense_row in dense_rows:
  for moe_row in moe_rows:
    if (
      dense_row.get("config_idx") == moe_row.get("config_idx")
      and dense_row.get("num_train_samples") == moe_row.get("num_train_samples")
      and dense_row.get("epochs") == moe_row.get("epochs")
    ):
      matched_pairs.append((dense_row, moe_row))

fig, ax = plt.subplots(figsize=(6, 5))
dense_loss = np.array([pair[0][moe_fit_target] for pair in matched_pairs], dtype=float)
moe_loss = np.array([pair[1][moe_fit_target] for pair in matched_pairs], dtype=float)
colors = np.array([pair[0]["num_train_samples"] for pair in matched_pairs], dtype=float)
scatter = ax.scatter(dense_loss, moe_loss, c=colors, cmap="viridis")
lo = min(dense_loss.min(), moe_loss.min())
hi = max(dense_loss.max(), moe_loss.max())
ax.plot([lo, hi], [lo, hi], color="black", linewidth=1)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("dense validation loss")
ax.set_ylabel("MoE validation loss")
ax.set_title(f"Matched dense vs MoE losses, epochs={moe_fit_epochs}")
fig.colorbar(scatter, ax=ax, label="train samples")
plt.tight_layout()
plt.show()


In [6]:
# TPU forward-pass timing harness for comparing baseline ragged_dot MoE vs fused Pallas MoE.
# The timing loop measures only forward_fn(batch) plus block_until_ready on the output.

import time


def make_timing_batches(
  *,
  num_batches=12,
  batch_size=512,
  num_samples=200_000,
  seed=2024,
):
  rng = jax.random.key(seed)
  rng, data_rng = jax.random.split(rng)
  x_data, y_data, loss_masks = make_addition_dataset(num_samples, data_rng)

  batches = []
  for _ in range(num_batches):
    rng, batch_rng = jax.random.split(rng)
    batch = get_batch(x_data, y_data, loss_masks, batch_size, batch_rng)
    # Put complete batches on device before timing so host transfer is excluded.
    batches.append(jax.device_put(batch))
  return batches


def block_until_ready_tree(value):
  return jax.block_until_ready(value)


def time_forward_pass(
  forward_fn,
  batches,
  *,
  warmup_batches=3,
  return_all_outputs=True,
):
  if not batches:
    raise ValueError("batches must be non-empty")

  warmup_count = min(warmup_batches, len(batches))
  for batch in batches[:warmup_count]:
    block_until_ready_tree(forward_fn(batch))

  times = []
  outputs = []
  for batch in batches:
    start = time.perf_counter()
    output = forward_fn(batch)
    output = block_until_ready_tree(output)
    elapsed = time.perf_counter() - start

    times.append(elapsed)
    if return_all_outputs:
      outputs.append(output)

  if not return_all_outputs:
    outputs = [output]

  return {
    "mean_seconds": float(sum(times) / len(times)),
    "min_seconds": float(min(times)),
    "max_seconds": float(max(times)),
    "num_batches": len(times),
    "per_batch_seconds": times,
    "outputs": outputs,
    "final_output": outputs[-1],
  }


def max_abs_diff_tree(a, b):
  leaves_a, treedef_a = jax.tree_util.tree_flatten(a)
  leaves_b, treedef_b = jax.tree_util.tree_flatten(b)
  if treedef_a != treedef_b:
    raise ValueError("output pytrees have different structures")
  return max(float(jnp.max(jnp.abs(x - y))) for x, y in zip(leaves_a, leaves_b))


def compare_output_lists(outputs_a, outputs_b):
  if len(outputs_a) != len(outputs_b):
    raise ValueError("output lists have different lengths")
  diffs = [max_abs_diff_tree(a, b) for a, b in zip(outputs_a, outputs_b)]
  return {
    "max_abs_diff": max(diffs),
    "mean_abs_diff_maxes": float(sum(diffs) / len(diffs)),
    "per_batch_max_abs_diff": diffs,
  }


def print_timing_result(name, timing):
  print(
    f"{name}: "
    f"mean={timing['mean_seconds'] * 1e3:.3f} ms, "
    f"min={timing['min_seconds'] * 1e3:.3f} ms, "
    f"max={timing['max_seconds'] * 1e3:.3f} ms "
    f"over {timing['num_batches']} batches"
  )


# Winning fused-kernel config: D=512, F=4096, E=4, top_k=1.
# Batch 512 gives M=B*T=6656 routed tokens; per-expert Pallas avoids materializing [M, F].
timing_base_config = GPTConfig(
  n_layer=6,
  n_head=8,
  n_embd=512,
  mlp_ratio=8,
  use_moe=True,
  use_fused_moe=False,
  num_experts=4,
  top_k=1,
)
timing_fused_config = GPTConfig(
  n_layer=timing_base_config.n_layer,
  n_head=timing_base_config.n_head,
  n_embd=timing_base_config.n_embd,
  dropout=timing_base_config.dropout,
  use_bias=timing_base_config.use_bias,
  mlp_ratio=timing_base_config.mlp_ratio,
  use_moe=True,
  use_fused_moe=True,
  fused_moe_backend="per_expert",
  num_experts=timing_base_config.num_experts,
  top_k=timing_base_config.top_k,
)

timing_baseline_model = GPT(timing_base_config)
timing_fused_model = GPT(timing_fused_config)
timing_batches = make_timing_batches(num_batches=12, batch_size=512, num_samples=200_000, seed=2024)

rng, timing_init_rng, timing_dropout_rng = jax.random.split(rng, 3)
timing_variables = timing_baseline_model.init(
  {"params": timing_init_rng, "dropout": timing_dropout_rng},
  timing_batches[0][0],
  train=False,
)

@jax.jit
def baseline_forward_fn(batch):
  x, _, _ = batch
  return timing_baseline_model.apply(timing_variables, x, train=False)


@jax.jit
def fused_forward_fn(batch):
  x, _, _ = batch
  return timing_fused_model.apply(timing_variables, x, train=False)


print("timing config:", timing_base_config)
print("num timing batches:", len(timing_batches), "batch shape:", timing_batches[0][0].shape)

baseline_timing = time_forward_pass(baseline_forward_fn, timing_batches, warmup_batches=3)
fused_timing = time_forward_pass(fused_forward_fn, timing_batches, warmup_batches=3)
comparison = compare_output_lists(baseline_timing["outputs"], fused_timing["outputs"])

print_timing_result("baseline ragged_dot", baseline_timing)
print_timing_result("fused pallas", fused_timing)
print(f"speedup: {baseline_timing['mean_seconds'] / fused_timing['mean_seconds']:.3f}x")
print("output comparison:", comparison)


timing config: GPTConfig(vocab_size=14, block_size=13, n_layer=6, n_head=8, n_embd=512, dropout=0.0, use_bias=True, mlp_ratio=8, dtype=<class 'jax.numpy.float32'>, use_moe=True, num_experts=4, top_k=1, use_fused_moe=False, fused_moe_backend='compact')
num timing batches: 12 batch shape: (512, 13)
baseline ragged_dot: mean=11.917 ms, min=11.877 ms, max=11.951 ms over 12 batches
fused pallas: mean=7.757 ms, min=7.722 ms, max=7.791 ms over 12 batches
speedup: 1.536x
output comparison: {'max_abs_diff': 0.0, 'mean_abs_diff_maxes': 0.0, 'per_batch_max_abs_diff': [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]}
